# Coding Nights 3.0 — Night 1, Part 2
### Two Pointers, Sliding Windows & the Interview Method
*Women in Tech × IEEE NTU Student Branch*

---

This notebook carries on from **`Night1_Part1_BigO_HashMaps.ipynb`**. Every pattern gets:

1. **The idea**: what the pattern is and when to use it
2. **The code**: a short, runnable solution
3. **A trace**: a version that prints each step as a table, so you can watch the algorithm work

| Topic | Pattern |
|---|---|
| Two Sum | HashMap / complement lookup |
| 🎮 Quiz | HashMaps & Sets |
| Two Pointers: When & Why | Two pointers |
| Converging pointers | Two pointers from opposite ends |
| Valid Palindrome | Converging pointers |
| Sliding Window | Fixed and variable windows |
| Longest Substring Without Repeats | Variable sliding window |
| In-Place Manipulation | Read / write pointers |
| Two Sum II (Sorted Array) | Converging pointers |
| 🎮 Quiz | Which pattern? |
| The 5-step interview method | Problem-solving process |
| ✏️ Practice problems | All of the above |

> **How to use this notebook:** run the **setup cell** first, then go top to bottom with `Shift + Enter`. Before you run a trace cell, try to **predict** what it will print. Quiz answers and solutions are hidden in **Reveal** boxes.

In [ ]:
# ── Setup: run this cell first ─────────────────────────────────────────
# Small display helpers so every result prints as a clean, scannable table.
# You don't need to understand this cell; just run it.
import math, random, time
from collections import Counter

LINE = 64

def banner(title):
    """Print a boxed section header."""
    print("═" * LINE)
    print(f"  {title}")
    print("═" * LINE)

def short(value, width=36):
    """repr() of a value, trimmed so long lists don't break the table."""
    text = repr(value)
    return text if len(text) <= width else text[: width - 1] + "…"

def show_table(rows, headers):
    """Print a list of rows as an aligned text table."""
    rows = [[str(cell) for cell in row] for row in rows]
    widths = [max([len(h)] + [len(r[i]) for r in rows]) for i, h in enumerate(headers)]
    fmt = lambda cells: " │ ".join(c.ljust(w) for c, w in zip(cells, widths))
    print(fmt(headers))
    print("─┼─".join("─" * w for w in widths))
    for row in rows:
        print(fmt(row))

def check(fn, cases, name=None):
    """Run fn(*args) for each (args, expected) pair and print a ✅/❌ table."""
    rows, passed = [], 0
    for i, (args, expected) in enumerate(cases, 1):
        shown_input = ", ".join(short(a, 30) for a in args)   # captured BEFORE fn can modify it
        try:
            got = fn(*args)
        except Exception as e:
            got = f"{type(e).__name__}: {e}"
        ok = got == expected
        passed += ok
        rows.append([i, shown_input, short(expected), short(got), "✅" if ok else "❌"])
    banner(f"Checking your answer: {name or fn.__name__}")
    show_table(rows, ["#", "input", "correct answer", "your answer", ""])
    print("─" * LINE)
    if all(row[3] == "None" for row in rows):
        print("⏳ Not attempted yet: replace `pass` with your code, run that cell, then re-run this one.")
    elif passed == len(cases):
        print(f"🎉 {passed}/{len(cases)} passed. All correct!")
    else:
        print(f"🔁 {passed}/{len(cases)} passed. Look at the ❌ rows and try again.")

print("✅ Setup complete: banner(), show_table() and check() are ready.")

---
## Problem: Two Sum

**What do we solve?**
Given an array `nums` and an integer `target`, return the **indices** of the two numbers that add up to `target`.

```
nums = [2, 7, 11, 15], target = 9   →   [0, 1]   (because 2 + 7 = 9)
```

### Recap: the complement idea
A **complement** is the value you still need to complete a condition:

$$\text{complement} = \text{target} - \text{current number}$$

If the target is 9 and the current number is 7, we need a **2**. Instead of searching the whole array for a 2, we ask: *have I already seen a 2?* A hash map answers that in **O(1)** time on average.

Three questions to ask yourself:
- What value would complete the target?
- Have I already seen that value?
- What information should I store with it? (Here we need the **index**.)

### Step 1: the brute-force baseline
Always start with the simplest correct solution, which here means trying every pair.

In [ ]:
def two_sum_brute(nums, target):
    n = len(nums)
    for i in range(n):
        for j in range(i + 1, n):      # every pair (i, j) with i < j
            if nums[i] + nums[j] == target:
                return [i, j]
    return []

banner("Brute force")
print("two_sum_brute([2, 7, 11, 15], 9) →", two_sum_brute([2, 7, 11, 15], 9))

This works, but it checks up to $\frac{n(n-1)}{2}$ pairs, which is **O(n²)** time. For 100,000 numbers that is about 5 billion checks.

### Step 2: the hash-map solution

**Algorithm**

For each number:
1. Calculate the complement: `target - number`
2. Check whether the complement is already in the dictionary
3. If it is, return both indices
4. Otherwise, store the current number and its index

The dictionary stores `number → index`, so we can find the partner without checking every pair.

In [ ]:
def two_sum(nums, target):
    seen = {}                          # number -> index where we saw it

    for i, num in enumerate(nums):
        needed = target - num          # the complement

        if needed in seen:             # O(1) average lookup
            return [seen[needed], i]

        seen[num] = i                  # remember this number for later

    return []                          # no pair found

### Trace: watch the dictionary grow

In [ ]:
def two_sum_trace(nums, target):
    """Same algorithm as two_sum, but prints a row for every step."""
    seen, rows, result = {}, [], []
    for i, num in enumerate(nums):
        needed = target - num
        found = needed in seen
        action = f"✔ found {needed} at index {seen[needed]}" if found else f"✘ not found, store {num}: {i}"
        rows.append([i, num, needed, short(dict(seen)), action])
        if found:
            result = [seen[needed], i]
            break
        seen[num] = i
    banner(f"Trace: two_sum({nums}, target={target})")
    show_table(rows, ["i", "num", "needed", "seen (before)", "action"])
    print(f"Result: {result}\n")
    return result

two_sum_trace([2, 7, 11, 15], 9)
two_sum_trace([3, 2, 4], 6);

**Why do we check *before* we store?** Look at `nums = [3, 3]`, `target = 6`. If we stored first, the first 3 would find *itself* as its own complement. Checking first means we only ever pair a number with an **earlier** number, never with itself.

### Complexity

| Approach | Time | Extra space |
|---|---|---|
| Brute force (every pair) | O(n²) | O(1) |
| Hash map (complement lookup) | **O(n)** | O(n) |

We **trade space for time**: we store up to n numbers so that each lookup is instant.

Let's measure the difference on a large input:

In [ ]:
random.seed(0)
n = 5_000
big = random.sample(range(1, 10_000_000), n)
target = big[-1] + big[-2]            # the answer is at the very end (worst case for brute force)

rows = []
for f, complexity in ((two_sum_brute, "O(n²)"), (two_sum, "O(n)")):
    start = time.perf_counter()
    f(big, target)
    rows.append([f.__name__, complexity, f"{time.perf_counter() - start:.4f}"])

banner(f"Race on n = {n:,} numbers")
show_table(rows, ["function", "time complexity", "seconds"])

---
## 🎮 Live Quiz: HashMaps & Sets

*(We play this live during the session. This is a copy you can keep and redo later.)* Write your answer down, then click **Reveal answer**.

**Q1.** A sign-up form must check whether a username is already taken, millions of times a day. Which structure should you use, and what does each check cost?
<details><summary>Reveal answer</summary>

A **set** of taken usernames. Each `name in taken` check is **O(1) on average**. A list would cost O(n) per check.
</details>

**Q2.** You need to know how many times every word appears in a document. Which pattern?
<details><summary>Reveal answer</summary>

A **frequency counter**: a `dict` (or `collections.Counter`) mapping word → count, built in one pass. **Time O(n), Space O(k)** for k distinct words.
</details>

**Q3.** What does `two_sum([1, 5, 3, 7], 8)` return? Trace it in your head first.
<details><summary>Reveal answer</summary>

**`[1, 2]`**. At i=0, 1 needs 7 (not seen, store `{1: 0}`). At i=1, 5 needs 3 (not seen, store `{1: 0, 5: 1}`). At i=2, 3 needs 5, **found at index 1**, so it returns `[1, 2]`. The pair 1 + 7 would also work, but we stop at the first pair we find.
</details>

**Q4.** In `two_sum`, why does `seen[num] = i` come **after** the `if needed in seen` check?
<details><summary>Reveal answer</summary>

So that a number never pairs with **itself**. With `nums = [4, 1]` and `target = 8`, storing first would find `8 − 4 = 4` straight away and wrongly return `[0, 0]`.
</details>

**Q5.** What are the time and space complexity of the hashmap Two Sum? And of the brute force?
<details><summary>Reveal answer</summary>

HashMap: **O(n) time, O(n) space.** Brute force: **O(n²) time, O(1) space.** This is the time-vs-space trade-off from Part 1.
</details>

---
## Two Pointers: When & Why

### What are two pointers?
Picture a row of objects. Instead of scanning the whole row again and again, you keep track of **two positions** (two indices) and move them in a smart way.

### When should I think about two pointers?
Look for a problem where:
1. You are working with an **array or a string**.
2. You need to **compare, search, or process elements at different positions**.
3. **Moving one pointer rules out work** you no longer need to do.

### Example
Find two numbers in a **sorted** array that add up to **6**:

```
target = 6
[1, 2, 3, 4, 6]
 L           R      1 + 6 = 7 > 6  → too big, move R left
 L        R         1 + 4 = 5 < 6  → too small, move L right
    L     R         2 + 4 = 6      → found it!
```

Each move throws away a whole set of pairs that we never need to check. That is where the speed comes from. Run the cell to watch it step by step:

In [ ]:
def show_pointers(arr, l, r):
    """Print the array with L and R markers under the two pointers."""
    print("   " + " ".join(f"{x:>3}" for x in arr))
    print("   " + " ".join(
        "L/R" if i == l == r else " L " if i == l else " R " if i == r else "   "
        for i in range(len(arr))
    ))

arr, target = [1, 2, 3, 4, 6], 6
l, r = 0, len(arr) - 1
banner(f"Two pointers on {arr}, target = {target}")
step = 1
while l < r:
    print(f"\nStep {step}:")
    show_pointers(arr, l, r)
    s = arr[l] + arr[r]
    if s == target:
        print(f"   {arr[l]} + {arr[r]} = {s} == {target} → found a pair! ✅")
        break
    elif s > target:
        print(f"   {arr[l]} + {arr[r]} = {s} > {target} → too big, move R left")
        r -= 1
    else:
        print(f"   {arr[l]} + {arr[r]} = {s} < {target} → too small, move L right")
        l += 1
    step += 1

---
## Pattern: Converging Pointers

The pointers start at **opposite ends** and move **towards each other** as we process the data.

```
 L →                     ← R
[ a , b , c , d , e , f , g ]
```

If a comparison tells us that some elements are **already dealt with** and can be ignored, we never look at them again. We move on.

### How to think about it
1. **What property does the problem give me?** (Sorted order? Symmetry?)
2. **What can I determine from the two ends?**
3. **What can I safely stop considering?**

Then move the pointers forward in their directions and repeat.

### A general template
Almost every converging-pointer solution has this shape:

In [ ]:
def converging_template(arr):
    l, r = 0, len(arr) - 1
    while l < r:                    # stop when the pointers meet
        # 1. look at arr[l] and arr[r]
        # 2. decide which side is "done" (or whether we found the answer)
        # 3. move l right, r left, or both
        l += 1
        r -= 1
    # 4. return the result

# Simplest example: reversing a list in place with converging pointers.
def reverse_in_place(arr):
    l, r = 0, len(arr) - 1
    while l < r:
        arr[l], arr[r] = arr[r], arr[l]   # swap the two ends
        l += 1
        r -= 1
    return arr

banner("reverse_in_place")
print("[1, 2, 3, 4, 5] →", reverse_in_place([1, 2, 3, 4, 5]))

---
## Problem: Valid Palindrome

**What is a palindrome?** A string that reads the same forwards and backwards.
- The first character must match the last.
- The second must match the second-last.
- …and so on.

This **symmetry from both ends** points straight to converging pointers.

> **In this version:** ignore anything that isn't a letter or digit, and ignore upper/lower case.
> `"A man, a plan, a canal: Panama"` → `True`

### Algorithm
1. Compare `s[l]` and `s[r]`.
2. If they **don't match**, it cannot be a palindrome. Return `False`.
3. If they **match**, that pair is fine. Move both pointers inward.
4. Continue until the pointers meet in the middle. If every pair matched, return `True`.

Extra rule for this problem: if a pointer is on a character that isn't alphanumeric (a space, comma, colon…), **skip it**.

### Code

In [ ]:
def is_palindrome(s):
    l, r = 0, len(s) - 1
    while l < r:
        if not s[l].isalnum(): l += 1                       # skip junk on the left
        elif not s[r].isalnum(): r -= 1                     # skip junk on the right
        elif s[l].lower() != s[r].lower(): return False     # mismatch: not a palindrome
        else: l += 1; r -= 1                                # match: move both inward
    return True

Every pass through the loop moves at least one pointer one step closer to the middle, so the loop runs at most n times. Let's trace it:

In [ ]:
def is_palindrome_trace(s):
    """Same algorithm as is_palindrome, but prints a row for every step."""
    l, r, rows, result = 0, len(s) - 1, [], True
    while l < r:
        a, b = s[l], s[r]
        if not a.isalnum():
            rows.append([l, r, repr(a), repr(b), "left is not alphanumeric → skip left"]); l += 1
        elif not b.isalnum():
            rows.append([l, r, repr(a), repr(b), "right is not alphanumeric → skip right"]); r -= 1
        elif a.lower() != b.lower():
            rows.append([l, r, repr(a), repr(b), "MISMATCH → return False ❌"]); result = False; break
        else:
            rows.append([l, r, repr(a), repr(b), "match → move both inward"]); l += 1; r -= 1
    banner(f"Trace: is_palindrome({s!r})")
    show_table(rows, ["l", "r", "s[l]", "s[r]", "decision"])
    print(f"Result: {result}" + ("  (pointers met, every pair matched ✅)" if result else "") + "\n")
    return result

is_palindrome_trace("Race, car!")
is_palindrome_trace("race a car");

### Complexity
- **Time: O(n).** Each character is visited at most once, by one of the two pointers.
- **Space: O(1).** We use two integers and never build a cleaned-up copy of the string.

Compare with the "obvious" Python one-liner, which is also O(n) but builds a new string (O(n) extra space):

In [ ]:
def is_palindrome_copy(s):
    cleaned = "".join(ch.lower() for ch in s if ch.isalnum())   # O(n) extra memory
    return cleaned == cleaned[::-1]

banner("is_palindrome vs. is_palindrome_copy")
show_table(
    [[repr(s), is_palindrome(s), is_palindrome_copy(s)] for s in ["A man, a plan, a canal: Panama", "race a car"]],
    ["string", "two pointers", "copy + reverse"],
)

> **Key takeaway:** if the problem is about **symmetry from both ends**, ask: *"Can I compare the two ends and work inward?"*

---
## Pattern: Sliding Window

What if we care about a **continuous section** of an array or string? Put a **window** around that section.

```
index:  0 1 2 3 4 5 6 7
s:      a b c a b c b b
          L   R            window = s[1..3] = "bca"
```

Two pointers mark the **left** and **right** edges of the window. Everything between them belongs to the current window.

Instead of recomputing everything for every possible section (O(n²) or worse), we **slide**: add the element that enters on the right and remove the element that leaves on the left.

### Two common types

| Type | Window size | How it moves |
|---|---|---|
| **Fixed window** | Always `k` | Add the new right element, remove the old left element |
| **Variable window** | Grows and shrinks | Expand right; **shrink left while the window is invalid** |

### Fixed window example: maximum sum of any `k` consecutive numbers

In [ ]:
def max_sum_k(nums, k):
    window_sum = sum(nums[:k])          # the first window
    best = window_sum
    for r in range(k, len(nums)):
        window_sum += nums[r]           # element entering on the right
        window_sum -= nums[r - k]       # element leaving on the left
        best = max(best, window_sum)
    return best

# Trace: slide a window of size k = 3 across nums
nums, k = [2, 1, 5, 1, 3, 2], 3
window_sum, best = sum(nums[:k]), sum(nums[:k])
rows = [[f"0..{k-1}", nums[:k], "—", "—", window_sum, best]]
for r in range(k, len(nums)):
    window_sum += nums[r] - nums[r - k]
    best = max(best, window_sum)
    rows.append([f"{r-k+1}..{r}", nums[r-k+1:r+1], f"+{nums[r]}", f"−{nums[r-k]}", window_sum, best])

banner(f"Fixed window, k = {k}, nums = {nums}")
show_table(rows, ["window", "elements", "enters", "leaves", "sum", "best so far"])
print(f"\nmax_sum_k → {max_sum_k(nums, k)}")

Each move of the window costs O(1) instead of re-summing `k` numbers, so the whole thing is **O(n)**.

The next problem uses a **variable-size** window.

---
## Problem: Longest Substring Without Repeating Characters

**What do we solve?** Given a string, find the length of the **longest substring** in which every character is different.

```
"abcabcbb"  →  3   ("abc")
"bbbbb"     →  1   ("b")
"pwwkew"    →  3   ("wke")   (not "pwke": the letters must be next to each other)
```

### Algorithm (variable window)
1. Start with an empty window. Add characters one at a time by moving `r` to the right.
2. If the window has **no repeats**, it is valid, so keep expanding.
3. If a repeat appears, the window is **invalid**. Do we start over? **No.** Move the left pointer forward until the duplicate has left the window and it is valid again.
4. After each step, **save the best answer so far**.

`Counter` (from `collections`) is just a tally of letters:

In [ ]:
banner('Counter("abca")')
tally = Counter("abca")
show_table([[ch, n] for ch, n in tally.items()], ["letter", "count"])

In [ ]:
def length_of_longest_substring(s):
    window = Counter()        # tally of characters inside s[l..r]
    l = 0
    best = 0
    for r, ch in enumerate(s):
        window[ch] += 1                   # expand: add s[r]
        while window[ch] > 1:             # invalid: ch now appears twice
            window[s[l]] -= 1             # shrink: remove s[l]
            l += 1
        best = max(best, r - l + 1)       # window is valid: record its length
    return best

In [ ]:
def longest_trace(s):
    """Same algorithm, but prints each step AND saves the best substring itself."""
    window, l, best, best_sub, rows = Counter(), 0, 0, "", []
    for r, ch in enumerate(s):
        window[ch] += 1
        dropped = ""
        while window[ch] > 1:              # shrink until ch is unique again
            dropped += s[l]
            window[s[l]] -= 1
            l += 1
        if r - l + 1 > best:               # save the substring, not just its length
            best, best_sub = r - l + 1, s[l:r + 1]
        rows.append([r, repr(ch), repr(dropped) if dropped else "—", repr(s[l:r + 1]), r - l + 1, repr(best_sub)])
    banner(f"Trace: longest substring without repeats in {s!r}")
    show_table(rows, ["r", "add", "dropped from left", "window", "length", "best substring"])
    print(f"Result: {best}  ({best_sub!r})\n")
    return best

longest_trace("abcabcbb")
longest_trace("pwwkew");

### Complexity
- **Time: O(n).** `r` moves forward n times and `l` moves forward at most n times in total. Even though there is a `while` inside the `for`, each character enters and leaves the window at most once.
- **Space: O(k)**, where k is the number of distinct characters we track (at most the size of the alphabet).

---
## Pattern: In-Place Manipulation

Sometimes we can solve a problem **inside the original array** instead of building a new copy.

**Why?** A second array uses O(n) extra memory. If we can safely modify or reuse the existing array, the extra space can drop to **O(1)**.

### How to think about it: read and write pointers
- The **read** pointer explores the array and looks at every element.
- The **write** pointer puts the result where it belongs, marking where the next "kept" element should go.

```
read:   moves every step
write:  moves only when we keep something
```

Because `write` never overtakes `read`, we never overwrite data we still need.

> *Try to use the space you already have.* Can you rearrange, overwrite or reuse the existing data instead of making a new copy?

### Example 1: Remove Duplicates from Sorted Array
Keep one copy of each value at the front of the array and return how many unique values there are.

In [ ]:
def remove_duplicates(nums):
    if not nums:
        return 0
    write = 1                                   # nums[0] is always kept
    for read in range(1, len(nums)):
        if nums[read] != nums[write - 1]:       # a new value we haven't kept yet
            nums[write] = nums[read]
            write += 1
    return write                                # the first `write` slots hold the answer

# Trace: watch read and write move
nums = [0, 0, 1, 1, 1, 2, 2, 3, 3, 4]
banner(f"Trace: remove_duplicates({nums})")
rows, write = [], 1
for read in range(1, len(nums)):
    keep = nums[read] != nums[write - 1]
    if keep:
        nums[write] = nums[read]
        write += 1
    rows.append([read, nums[read], "keep → write it" if keep else "duplicate → skip", write, nums[:write]])
show_table(rows, ["read", "value", "decision", "write (after)", "kept so far"])
print(f"\nk = {write}, nums[:k] = {nums[:write]}, whole array = {nums}")

Note that the slots after `k` still hold leftover values. The problem only asks about the first `k` slots, so that is fine.

### Example 2: Move Zeroes
Move all zeros to the end while keeping the order of the non-zero elements. We swap instead of overwrite, so no values are lost.

In [ ]:
def move_zeroes(nums):
    write = 0
    for read in range(len(nums)):
        if nums[read] != 0:
            nums[write], nums[read] = nums[read], nums[write]
            write += 1
    return nums

def move_zeroes_trace(nums):
    """Same algorithm, but prints a row for every step."""
    banner(f"Trace: move_zeroes({nums})")
    write, rows = 0, []
    for read in range(len(nums)):
        value = nums[read]
        if value != 0:
            nums[write], nums[read] = nums[read], nums[write]
            write += 1
        rows.append([read, value, "non-zero → swap to write" if value != 0 else "zero → skip", write, list(nums)])
    show_table(rows, ["read", "value", "decision", "write (after)", "array"])
    return nums

move_zeroes_trace([0, 1, 0, 3, 12]);

Both examples run in **O(n) time and O(1) extra space**.

---
## Problem: Two Sum II (Sorted Array)

**What do we solve?** Find two numbers that add up to `target`, but this time the array is **sorted**. We return **positions counting from 1** (not the usual indices that start at 0).

```
numbers = [2, 7, 11, 15], target = 9   →   [1, 2]
```

Because the array is sorted, we can do much better than trying every pair, and we don't need a hash map either.

### Algorithm
Start at the two opposite ends and look at the sum. The sorted order tells us what to do:
- **Sum too small?** We need a bigger number, so move `l` **right**.
- **Sum too large?** We need a smaller number, so move `r` **left**.
- **Sum equal?** Done.

### Why is it safe to throw elements away?
Suppose `numbers[l] + numbers[r] < target`. `numbers[r]` is the **largest** value left, so `numbers[l]` plus *any* remaining value is also too small. `numbers[l]` can never be part of the answer, so we discard it with `l += 1`. The same logic, mirrored, applies to `r`.

### Code

In [ ]:
def two_sum_sorted(numbers, target):
    l, r = 0, len(numbers) - 1
    while l < r:
        s = numbers[l] + numbers[r]
        if s == target: return [l + 1, r + 1]   # +1: positions count from 1
        elif s < target: l += 1                 # too small: need a bigger left value
        else: r -= 1                            # too large: need a smaller right value
    return []

In [ ]:
def two_sum_sorted_trace(numbers, target):
    """Same algorithm, but draws the pointers at every step."""
    banner(f"Trace: two_sum_sorted({numbers}, target={target})")
    l, r, step = 0, len(numbers) - 1, 1
    while l < r:
        s = numbers[l] + numbers[r]
        print(f"\nStep {step}:")
        show_pointers(numbers, l, r)
        if s == target:
            print(f"   {numbers[l]} + {numbers[r]} = {s} → found! return {[l + 1, r + 1]} ✅")
            return [l + 1, r + 1]
        elif s < target:
            print(f"   {numbers[l]} + {numbers[r]} = {s} < {target} → too small, move L right"); l += 1
        else:
            print(f"   {numbers[l]} + {numbers[r]} = {s} > {target} → too large, move R left");  r -= 1
        step += 1
    return []

two_sum_sorted_trace([1, 3, 4, 5, 7, 10, 11], 9);

### Complexity and comparison

| | Two Sum | Two Sum II |
|---|---|---|
| Input | unsorted | **sorted** |
| Pattern | hash map | converging pointers |
| Time | O(n) | O(n) |
| Extra space | O(n) | **O(1)** |

> **Key takeaway:** when you see *"find a pair in a **sorted** array that satisfies some condition"*, ask: *"Can I use the order to eliminate possibilities?"*

---
## 🎮 Live Quiz: Which pattern?

*(Played live during the session. Here is a copy to keep.)* For each problem, name the pattern **and** the time complexity before you reveal the answer.

**Q1.** Given an array of **positive** numbers, find the length of the longest **contiguous** subarray whose sum is ≤ `k`.
<details><summary>Reveal answer</summary>

**Variable sliding window.** Expand right; while the sum is > k, shrink from the left. **O(n)** time, O(1) space. (Positive numbers matter: adding an element can only increase the sum.)
</details>

**Q2.** Check whether a string reads the same forwards and backwards.
<details><summary>Reveal answer</summary>

**Converging pointers** (Valid Palindrome). **O(n)** time, O(1) space.
</details>

**Q3.** Remove every occurrence of `val` from an array **without** creating a new array.
<details><summary>Reveal answer</summary>

**In-place read/write pointers**. **O(n)** time, **O(1)** extra space. You'll write this in Practice Exercise 4.
</details>

**Q4.** Two Sum on an **unsorted** array, returning the **original** indices. Should you sort first and use two pointers?
<details><summary>Reveal answer</summary>

**No: use the HashMap.** Sorting costs O(n log n) and shuffles the indices you need to return. The hashmap version is O(n).
</details>

**Q5.** `length_of_longest_substring` has a `while` loop inside a `for` loop. Why is it O(n) and not O(n²)?
<details><summary>Reveal answer</summary>

Each character **enters** the window once (when `r` passes it) and **leaves** at most once (when `l` passes it). `l` and `r` each move at most n steps **in total**, so there are ≤ 2n moves overall, which is **O(n)**.
</details>

**Q6.** What does `two_sum_sorted([1, 2, 4, 7, 11], 15)` return?
<details><summary>Reveal answer</summary>

**`[3, 5]`**. 1+11=12 < 15, so L moves right. 2+11=13 < 15, so L moves right. 4+11=15, found at positions 3 and 5 (counting from 1).
</details>

---
## Live Interview Breakdown: The 5-Step Method

| Step | What to do |
|---|---|
| **1. Clarify the problem** | Make sure you understand the input and what the function should return. |
| **2. Start with brute force** | Describe the simplest correct approach. Then identify what makes it slow. |
| **3. Find the pattern** | Ask: can a hash map/set, two pointers, or another structure remove repeated work? |
| **4. Code and explain** | Write clean code while explaining your choices and the complexity. |
| **5. Test and analyse** | Dry-run an example by hand, then state the time and space complexity. |

### Worked example: Contains Duplicate
> Given an integer array `nums`, return `True` if any value appears at least twice.

**1. Clarify.** What do we return: `True`/`False`, or the duplicate itself? (`True`/`False`.) How big can the input be? (Large, so O(n²) is too slow.)

**2. Brute force.** Compare every pair: O(n²) time. It is slow because, for each element, we **re-scan** everything before it to ask *"have I seen this before?"*.

In [ ]:
def contains_duplicate_brute(nums):
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            if nums[i] == nums[j]:
                return True
    return False

**3. Find the pattern.** *"Have I seen this before?"* is the signal for a **hash set**: O(1) average membership checks.

**4. Code and explain.** Key idea: at every step, `seen` holds exactly the numbers we have already looked at.

In [ ]:
def contains_duplicate(nums):
    seen = set()
    for num in nums:
        if num in seen:        # seen before -> duplicate
            return True
        seen.add(num)
    return False               # loop finished: every value was unique

**5. Test and analyse.** Dry-run an example by hand, then run it:

In [ ]:
banner("contains_duplicate")
show_table([[nums, contains_duplicate(nums)] for nums in ([1, 2, 3, 1], [1, 2, 3, 4])], ["nums", "result"])

**Complexity:** O(n) time, O(n) space. A one-liner alternative is `len(set(nums)) != len(nums)`. It has the same complexity, but it can't stop early when it finds a duplicate.

---
## ✏️ Practice problems

Each exercise has four parts:

1. **Problem**: what to solve, with examples.
2. **Your code**: replace `pass` with your solution and run the cell.
3. **Check cell**: run it to compare your answers with the correct ones (✅ / ❌).
4. **Reveal answer key**: a fully explained solution. Give the problem a real try first!

Try the **5-step method** on each one: clarify → brute force → pattern → code → test.

| # | Problem | Pattern | Difficulty |
|---|---|---|---|
| 1 | Squares of a Sorted Array | Converging pointers | 🟢 Easy |
| 2 | Maximum Average Subarray | Fixed sliding window | 🟢 Easy |
| 3 | Minimum Size Subarray Sum | Variable sliding window | 🟡 Medium |
| 4 | Remove Element | In-place read/write pointers | 🟢 Easy |
| 5 | Container With Most Water | Converging pointers + 5-step method | 🟡 Medium |

### Exercise 1 — Squares of a Sorted Array 🟢 *(Converging pointers)*

Given an array sorted from smallest to largest, which **may contain negative numbers**, return the squares of each number, **also sorted**.


| Input | Output |
|---|---|
| `[-4, -1, 0, 3, 10]` | `[0, 1, 9, 16, 100]` |
| `[-7, -3, 2, 3, 11]` | `[4, 9, 9, 49, 121]` |

*Hint: where is the **largest** square always found? Compare the two ends and fill the result from the back.*

In [ ]:
def sorted_squares(nums):
    """
    Return the squares of a sorted list, in sorted order.

    Example: sorted_squares([-4, -1, 0, 3, 10])  →  [0, 1, 9, 16, 100]
    Pattern: Converging pointers. Target: O(n) time.
    """
    pass  # ✏️ replace with your code

In [ ]:
check(sorted_squares, [
    (([-4, -1, 0, 3, 10],), [0, 1, 9, 16, 100]),
    (([-7, -3, 2, 3, 11],), [4, 9, 9, 49, 121]),
    (([1, 2, 3],), [1, 4, 9]),
    (([-3, -2, -1],), [1, 4, 9]),
    (([],), []),
])

<details><summary>Reveal answer key</summary>

```python
def sorted_squares(nums):
    n = len(nums)
    result = [0] * n
    l, r = 0, n - 1

    # The biggest square is always at one of the two ENDS: a very negative
    # number on the left or a big positive number on the right.
    # So we fill result from the back, largest square first.
    for write in range(n - 1, -1, -1):
        if abs(nums[l]) > abs(nums[r]):
            result[write] = nums[l] ** 2   # left end wins: use it, move l inward
            l += 1
        else:
            result[write] = nums[r] ** 2   # right end wins (or tie): use it, move r inward
            r -= 1
    return result
```
**Time O(n)**: each element is placed exactly once. **Space O(n)** for the output (O(1) extra apart from that).
The property the problem gives us is **sorted order**. What we can determine from the two ends is **which one has the bigger square**.

</details>

### Exercise 2 — Maximum Average Subarray 🟢 *(Fixed sliding window)*

Given `nums` and an integer `k`, return the **maximum average** of any **contiguous** subarray of length exactly `k`.


| `nums` | `k` | Output | Why |
|---|---|---|---|
| `[1, 12, -5, -6, 50, 3]` | `4` | `12.75` | (12 − 5 − 6 + 50) / 4 |
| `[5]` | `1` | `5.0` | |

*Hint: this is `max_sum_k` from the Sliding Window section in disguise.*

In [ ]:
def find_max_average(nums, k):
    """
    Return the maximum average of any length-k contiguous subarray.

    Example: find_max_average([1, 12, -5, -6, 50, 3], 4)  →  12.75
    Pattern: Fixed sliding window. Target: O(n) time, O(1) space.
    """
    pass  # ✏️ replace with your code

In [ ]:
check(find_max_average, [
    (([1, 12, -5, -6, 50, 3], 4), 12.75),
    (([5], 1), 5.0),
    (([0, 4, 0, 3, 2], 1), 4.0),
    (([-1, -2, -3], 2), -1.5),
])

<details><summary>Reveal answer key</summary>

```python
def find_max_average(nums, k):
    # Sum of the first window: nums[0..k-1]
    window_sum = sum(nums[:k])
    best = window_sum

    # Slide the window one step at a time:
    # nums[r] enters on the right, nums[r - k] leaves on the left.
    for r in range(k, len(nums)):
        window_sum += nums[r] - nums[r - k]
        best = max(best, window_sum)

    # The best average belongs to the best sum, because k is fixed.
    return best / k
```
**Time O(n), Space O(1).** Re-summing every window would be O(n·k).

</details>

### Exercise 3 — Minimum Size Subarray Sum 🟡 *(Variable sliding window)*

Given a `target` and an array of **positive** integers, return the **length of the shortest contiguous subarray** whose sum is **≥ target**. If there isn't one, return `0`.


| `target` | `nums` | Output | Why |
|---|---|---|---|
| `7` | `[2, 3, 1, 2, 4, 3]` | `2` | `[4, 3]` |
| `4` | `[1, 4, 4]` | `1` | `[4]` |
| `11` | `[1, 1, 1, 1, 1, 1, 1, 1]` | `0` | the total is only 8 |

*Hint: expand right until the window is "good" (sum ≥ target), then shrink from the left **while it stays good**, recording the length each time.*

In [ ]:
def min_subarray_len(target, nums):
    """
    Return the length of the shortest contiguous subarray with sum >= target, or 0.

    Example: min_subarray_len(7, [2, 3, 1, 2, 4, 3])  →  2
    Pattern: Variable sliding window. Target: O(n) time, O(1) space.
    """
    pass  # ✏️ replace with your code

In [ ]:
check(min_subarray_len, [
    ((7, [2, 3, 1, 2, 4, 3]), 2),
    ((4, [1, 4, 4]), 1),
    ((11, [1, 1, 1, 1, 1, 1, 1, 1]), 0),
    ((15, [1, 2, 3, 4, 5]), 5),
    ((3, []), 0),
])

<details><summary>Reveal answer key</summary>

```python
def min_subarray_len(target, nums):
    l = 0
    window_sum = 0
    best = float("inf")          # "no answer yet"

    for r in range(len(nums)):
        window_sum += nums[r]                    # expand: add nums[r]

        # While the window is good, record it, then try to make it shorter.
        while window_sum >= target:
            best = min(best, r - l + 1)
            window_sum -= nums[l]                # shrink: drop nums[l]
            l += 1

    return 0 if best == float("inf") else best
```
**Time O(n)**: like Longest Substring, `l` and `r` each move at most n times in total. **Space O(1).**
Note the flipped logic compared with Longest Substring: there we shrank while the window was **invalid**; here we shrink while it is **valid**, because we want the *shortest* window.

</details>

### Exercise 4 — Remove Element 🟢 *(In-place read/write pointers)*

Remove every occurrence of `val` from `nums` **in place**. Return `k`, the number of elements that are **not** `val`. The first `k` slots of `nums` must hold those elements (in any order). What's left after slot `k` doesn't matter.


| `nums` | `val` | Return `k` | `nums[:k]` (any order) |
|---|---|---|---|
| `[3, 2, 2, 3]` | `3` | `2` | `[2, 2]` |
| `[0, 1, 2, 2, 3, 0, 4, 2]` | `2` | `5` | `[0, 1, 3, 0, 4]` |

*Hint: it's the same shape as `remove_duplicates` from the In-Place section. The only thing that changes is the "keep it?" test.*

In [ ]:
def remove_element(nums, val):
    """
    Move every element != val to the front of nums (in place) and return how many there are.

    Example: nums = [3, 2, 2, 3], val = 3  →  returns 2, and nums starts with [2, 2]
    Pattern: In-place read/write pointers. Target: O(n) time, O(1) extra space.
    """
    pass  # ✏️ replace with your code

In [ ]:
def remove_element_result(nums, val):
    """Runs your function and shows k and the kept numbers (sorted, because order doesn't matter)."""
    k = remove_element(nums, val)
    return None if k is None else (k, sorted(nums[:k]))

check(remove_element_result, [
    (([3, 2, 2, 3], 3), (2, [2, 2])),
    (([0, 1, 2, 2, 3, 0, 4, 2], 2), (5, [0, 0, 1, 3, 4])),
    (([1], 1), (0, [])),
    (([4, 5], 9), (2, [4, 5])),
    (([], 0), (0, [])),
], name="remove_element")

<details><summary>Reveal answer key</summary>

```python
def remove_element(nums, val):
    write = 0                         # next slot for an element we keep
    for read in range(len(nums)):     # read explores every element
        if nums[read] != val:         # keep it?
            nums[write] = nums[read]  # put it where it belongs
            write += 1
        # else: skip it. write stays put, so this slot gets overwritten later.
    return write                      # nums[:write] holds every kept element
```
**Time O(n), Space O(1).** Since `write ≤ read` at all times, we never overwrite an element we haven't read yet.

</details>

### Exercise 5 — Container With Most Water 🟡 *(Converging pointers + the 5-step method)*

`height[i]` is the height of a vertical line at position `i`. Pick two lines that, together with the x-axis, form a container. Return the **maximum amount of water** it can hold.
The water between lines `i` and `j` is `min(height[i], height[j]) × (j − i)`.


| Input | Output | Why |
|---|---|---|
| `[1, 8, 6, 2, 5, 4, 8, 3, 7]` | `49` | lines at index 1 (8) and 8 (7): min(8, 7) × 7 |
| `[1, 1]` | `1` | |

**Walk through the 5 steps before you code:**
1. **Clarify**: do we return the amount of water or the positions of the lines?
2. **Brute force**: try every pair `(i, j)`. What does that cost?
3. **Pattern**: start with the **widest** container (`l = 0`, `r = n − 1`). Which pointer should move, and **why is it safe** to discard that line?
4. **Code and explain.**
5. **Test**: dry-run `[1, 8, 6, 2, 5, 4, 8, 3, 7]` by hand.

In [ ]:
def max_area(height):
    """
    Return the maximum water held between any two lines.

    Example: max_area([1, 8, 6, 2, 5, 4, 8, 3, 7])  →  49
    Pattern: Converging pointers. Target: O(n) time, O(1) space.
    """
    pass  # ✏️ replace with your code

In [ ]:
check(max_area, [
    (([1, 8, 6, 2, 5, 4, 8, 3, 7],), 49),
    (([1, 1],), 1),
    (([4, 3, 2, 1, 4],), 16),
    (([1, 2, 1],), 2),
    (([0, 0],), 0),
])

<details><summary>Reveal answer key</summary>

**5-step notes**
1. *Clarify:* return the amount of water (a number).
2. *Brute force:* check all pairs, which is O(n²) and too slow for big inputs.
3. *Pattern:* converging pointers. The area is limited by the **shorter** line. If we moved the taller line inward, the width would shrink and the height could **never** go above the shorter line, so the area can't improve. The shorter line has therefore been fully "dealt with", and we can safely discard it.

```python
def max_area(height):
    l, r = 0, len(height) - 1
    best = 0
    while l < r:
        # Water is capped by the shorter line, times the distance between the lines.
        area = min(height[l], height[r]) * (r - l)
        best = max(best, area)

        # Discard the shorter line: no container using it can beat `area`.
        if height[l] < height[r]:
            l += 1
        else:
            r -= 1
    return best
```
5. *Analyse:* **Time O(n)**, because each step moves one pointer inward. **Space O(1).**

</details>

---
## Practice Roadmap: NeetCode 75 + Weekly Habit

- Do **at least one topic every week** (the roadmap is at neetcode.io).
- When you feel ready, attempt the **Blind 75** on NeetCode: <https://neetcode.io/practice/practice/blind75>

Suggested routine for each problem:
1. Spend 15–20 minutes on it yourself, using the 5-step method.
2. If you're stuck, look at the *pattern* only (not the code) and try again.
3. After solving it, write down **in one sentence** which signal pointed to the pattern.
4. Solve it again from scratch a few days later.

---
### Summary

| Pattern | Signal in the problem | Typical cost |
|---|---|---|
| **Hash map / set** | "have I seen…?", "count…", "find the partner…" | O(n) time, O(n) space |
| **Converging pointers** | sorted order, or symmetry from both ends | O(n) time, O(1) space |
| **Sliding window** | "contiguous subarray / substring" | O(n) time, O(1)–O(k) space |
| **Read/write pointers** | "in place", "without extra memory" | O(n) time, O(1) space |
| **Interview method** | every problem! | clarify → brute force → pattern → code & explain → test & analyse |

**Happy coding, and see you at Night 2!** 🌙